In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.imports as lib

In [2]:
# Đọc dữ liệu Excel
file_path = "../data/data-clean/du_lieu_that.csv"
raw_df = lib.pd.read_csv(file_path)

choice_columns = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]
dfs = {
    f"Diem_L{grade}": raw_df[
        ["ma_hoc_sinh"]
        + [col for col in raw_df.columns if col.startswith(f"L{grade}_")]
        + choice_columns
    ].copy()
    for grade in [6, 7, 8, 9]
}

In [3]:
# Kiểm tra số lượng và tỷ lệ Missing Values trong từng sheet

for sheet_name, df in dfs.items():
    missing_count = df.isna().sum()
    missing_percent = (missing_count / len(df) * 100).round(2)

    missing_report = lib.pd.DataFrame({
        "Số lượng thiếu": missing_count,
        "Tỷ lệ thiếu (%)": missing_percent
    })

    # Chỉ hiển thị các cột có dữ liệu thiếu
    missing_report = missing_report[
        missing_report["Số lượng thiếu"] > 0
    ].sort_values("Tỷ lệ thiếu (%)", ascending=False)

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")
    print(f"Tổng số ô thiếu: {df.isna().sum().sum():,}")
    print(f"Số cột có dữ liệu thiếu: {(missing_count > 0).sum()}")
    print(f"Tỷ lệ thiếu toàn bộ: {df.isna().mean().mean() * 100:.2f}%")

    if missing_report.empty:
        print("PASS: Không có dữ liệu thiếu.")
    else:
        display(missing_report)


SHEET: Diem_L6
Tổng số ô thiếu: 0
Số cột có dữ liệu thiếu: 0
Tỷ lệ thiếu toàn bộ: 0.00%
PASS: Không có dữ liệu thiếu.

SHEET: Diem_L7
Tổng số ô thiếu: 0
Số cột có dữ liệu thiếu: 0
Tỷ lệ thiếu toàn bộ: 0.00%
PASS: Không có dữ liệu thiếu.

SHEET: Diem_L8
Tổng số ô thiếu: 0
Số cột có dữ liệu thiếu: 0
Tỷ lệ thiếu toàn bộ: 0.00%
PASS: Không có dữ liệu thiếu.

SHEET: Diem_L9
Tổng số ô thiếu: 0
Số cột có dữ liệu thiếu: 0
Tỷ lệ thiếu toàn bộ: 0.00%
PASS: Không có dữ liệu thiếu.


In [4]:
# Kiểm tra dòng trùng lặp và mã học sinh trùng lặp trong từng sheet

id_col = "ma_hoc_sinh"

for sheet_name, df in dfs.items():

    # Kiểm tra dòng trùng hoàn toàn
    duplicate_rows = df.duplicated().sum()

    # Kiểm tra mã học sinh trùng (không tính giá trị thiếu)
    duplicate_ids = df.loc[
        df[id_col].notna(), id_col
    ].duplicated().sum()

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")

    print(f"Tổng số dòng: {len(df):,}")
    print(f"Số dòng trùng hoàn toàn: {duplicate_rows:,}")
    print(f"Tỷ lệ dòng trùng: {duplicate_rows / len(df) * 100:.2f}%")

    print(f"\nSố mã học sinh trùng: {duplicate_ids:,}")
    print(f"Tỷ lệ mã trùng: {duplicate_ids / len(df) * 100:.2f}%")

    if duplicate_rows == 0 and duplicate_ids == 0:
        print("\nPASS: Không phát hiện dữ liệu trùng lặp.")
    else:
        print("\nFAIL: Phát hiện dữ liệu trùng lặp.")

        # Hiển thị toàn bộ dòng có mã học sinh trùng
        duplicated_data = df[
            df[id_col].notna()
            & df[id_col].duplicated(keep=False)
        ]

        if not duplicated_data.empty:
            display(duplicated_data.sort_values(id_col))


SHEET: Diem_L6
Tổng số dòng: 130
Số dòng trùng hoàn toàn: 0
Tỷ lệ dòng trùng: 0.00%

Số mã học sinh trùng: 0
Tỷ lệ mã trùng: 0.00%

PASS: Không phát hiện dữ liệu trùng lặp.

SHEET: Diem_L7
Tổng số dòng: 130
Số dòng trùng hoàn toàn: 0
Tỷ lệ dòng trùng: 0.00%

Số mã học sinh trùng: 0
Tỷ lệ mã trùng: 0.00%

PASS: Không phát hiện dữ liệu trùng lặp.

SHEET: Diem_L8
Tổng số dòng: 130
Số dòng trùng hoàn toàn: 0
Tỷ lệ dòng trùng: 0.00%

Số mã học sinh trùng: 0
Tỷ lệ mã trùng: 0.00%

PASS: Không phát hiện dữ liệu trùng lặp.

SHEET: Diem_L9
Tổng số dòng: 130
Số dòng trùng hoàn toàn: 0
Tỷ lệ dòng trùng: 0.00%

Số mã học sinh trùng: 0
Tỷ lệ mã trùng: 0.00%

PASS: Không phát hiện dữ liệu trùng lặp.


In [5]:
# Kiểm tra điểm ngoài phạm vi hợp lệ [0, 10]

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

for sheet_name, df in dfs.items():

    # Chỉ lấy các cột điểm số
    score_cols = df.select_dtypes(include=["number"]).columns.difference(choice_cols)

    # Phát hiện giá trị không hợp lệ
    invalid_mask = (df[score_cols] < 0) | (df[score_cols] > 10)

    invalid_count = invalid_mask.sum()
    invalid_report = lib.pd.DataFrame({
        "Số giá trị không hợp lệ": invalid_count,
        "Tỷ lệ không hợp lệ (%)": (
            invalid_count / len(df) * 100
        ).round(2)
    })

    invalid_report = invalid_report[
        invalid_report["Số giá trị không hợp lệ"] > 0
    ]

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")

    print(f"Số cột điểm được kiểm tra: {len(score_cols)}")
    print(f"Tổng giá trị không hợp lệ: {invalid_mask.sum().sum():,}")
    print(f"Số học sinh có điểm không hợp lệ: {invalid_mask.any(axis=1).sum():,}")

    if invalid_report.empty:
        print("\nPASS: Tất cả điểm nằm trong phạm vi [0, 10].")
    else:
        print("\nFAIL: Phát hiện điểm ngoài phạm vi hợp lệ.")
        display(invalid_report)

        # Hiển thị học sinh có điểm không hợp lệ
        display(
            df.loc[
                invalid_mask.any(axis=1),
                [id_col] + list(score_cols)
            ]
        )


SHEET: Diem_L6
Số cột điểm được kiểm tra: 25
Tổng giá trị không hợp lệ: 130
Số học sinh có điểm không hợp lệ: 130

FAIL: Phát hiện điểm ngoài phạm vi hợp lệ.


,Số giá trị không hợp lệ,Tỷ lệ không hợp lệ (%)
ma_hoc_sinh,130,100.0


,ma_hoc_sinh,L6_cong_nghe_ca_nam,L6_cong_nghe_hki,L6_cong_nghe_hkii,L6_gdcd_ca_nam,L6_gdcd_hki,L6_gdcd_hkii,L6_khtn_ca_nam,L6_khtn_hki,L6_khtn_hkii,...,L6_tin_hoc_ca_nam,L6_tin_hoc_hki,L6_tin_hoc_hkii,L6_toan_ca_nam,L6_toan_hki,L6_toan_hkii,L6_van_ca_nam,L6_van_hki,L6_van_hkii,ma_hoc_sinh
0,8000180385,9.4,9.4,9.4,8.5,8.9,8.3,8.5,8.6,8.5,...,8.4,7.8,8.7,8.1,8.7,7.8,8.5,8.2,8.7,8000180385
1,8000180391,7.5,8.1,7.2,8.0,8.1,8.0,6.6,6.9,6.5,...,5.1,5.6,4.9,6.0,5.5,6.3,6.3,6.4,6.3,8000180391
2,8000180790,9.6,9.9,9.4,8.8,9.7,8.4,8.9,8.9,8.9,...,8.8,9.4,8.5,7.7,7.0,8.1,8.9,8.8,9.0,8000180790
3,8000180756,9.3,9.4,9.2,8.5,7.4,9.0,8.3,8.8,8.0,...,9.1,8.7,9.3,7.3,7.4,7.3,8.4,7.7,8.7,8000180756
4,8000180587,9.7,9.6,9.7,9.5,9.1,9.7,9.6,9.7,9.6,...,8.4,9.0,8.1,8.9,8.8,8.9,9.6,9.7,9.6,8000180587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,8000180818,9.4,9.5,9.4,9.2,9.1,9.3,8.9,8.4,9.1,...,8.1,9.0,7.7,6.6,4.5,7.6,7.9,7.9,7.9,8000180818
126,8000180545,7.7,8.6,7.2,7.6,7.9,7.4,7.7,7.8,7.6,...,5.9,6.4,5.6,7.4,7.5,7.4,6.9,7.3,6.7,8000180545
127,8000180738,7.6,7.7,7.6,6.2,5.0,6.8,6.4,6.3,6.5,...,5.4,5.0,5.6,5.4,5.4,5.4,6.1,5.6,6.3,8000180738
128,8000180550,9.5,9.7,9.4,8.8,7.5,9.4,8.7,8.8,8.6,...,8.3,9.1,7.9,8.3,8.7,8.1,9.4,9.1,9.6,8000180550



SHEET: Diem_L7
Số cột điểm được kiểm tra: 25
Tổng giá trị không hợp lệ: 130
Số học sinh có điểm không hợp lệ: 130

FAIL: Phát hiện điểm ngoài phạm vi hợp lệ.


,Số giá trị không hợp lệ,Tỷ lệ không hợp lệ (%)
ma_hoc_sinh,130,100.0


,ma_hoc_sinh,L7_cong_nghe_ca_nam,L7_cong_nghe_hki,L7_cong_nghe_hkii,L7_gdcd_ca_nam,L7_gdcd_hki,L7_gdcd_hkii,L7_khtn_ca_nam,L7_khtn_hki,L7_khtn_hkii,...,L7_tin_hoc_ca_nam,L7_tin_hoc_hki,L7_tin_hoc_hkii,L7_toan_ca_nam,L7_toan_hki,L7_toan_hkii,L7_van_ca_nam,L7_van_hki,L7_van_hkii,ma_hoc_sinh
0,8000180385,9.4,8.6,9.8,9.0,8.4,9.3,8.2,8.8,7.9,...,7.6,7.5,7.6,7.8,7.2,8.1,7.9,8.4,7.6,8000180385
1,8000180391,6.4,5.6,6.8,7.1,8.2,6.5,6.5,6.8,6.3,...,6.1,6.3,6.0,3.9,3.8,4.0,6.2,5.7,6.4,8000180391
2,8000180790,9.1,8.5,9.4,9.1,8.8,9.2,8.3,9.0,8.0,...,7.3,8.0,7.0,7.7,8.1,7.5,8.9,9.4,8.6,8000180790
3,8000180756,8.8,8.1,9.1,8.7,8.0,9.1,8.6,8.4,8.7,...,7.2,7.9,6.9,7.5,7.8,7.3,8.8,8.5,8.9,8000180756
4,8000180587,9.4,9.1,9.6,9.4,9.3,9.5,9.5,9.7,9.4,...,8.1,7.6,8.4,8.9,8.9,8.9,8.9,8.8,9.0,8000180587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,8000180818,9.3,8.8,9.6,8.1,7.1,8.6,7.8,7.7,7.8,...,6.9,6.9,6.9,6.0,6.9,5.5,7.6,8.0,7.4,8000180818
126,8000180545,7.9,7.5,8.1,7.5,7.6,7.4,6.9,7.8,6.4,...,6.8,5.9,7.3,4.7,4.4,4.8,7.3,7.6,7.1,8000180545
127,8000180738,7.8,7.7,7.8,7.3,7.4,7.3,7.2,7.1,7.2,...,6.0,6.1,6.0,5.7,5.4,5.8,6.1,5.6,6.3,8000180738
128,8000180550,9.4,8.7,9.7,9.6,9.5,9.6,8.8,8.8,8.8,...,7.5,7.5,7.5,7.2,8.0,6.8,8.2,8.1,8.2,8000180550


... (2 outputs left out)

,ma_hoc_sinh,L8_cong_nghe_ca_nam,L8_cong_nghe_hki,L8_cong_nghe_hkii,L8_gdcd_ca_nam,L8_gdcd_hki,L8_gdcd_hkii,L8_khtn_ca_nam,L8_khtn_hki,L8_khtn_hkii,...,L8_tin_hoc_ca_nam,L8_tin_hoc_hki,L8_tin_hoc_hkii,L8_toan_ca_nam,L8_toan_hki,L8_toan_hkii,L8_van_ca_nam,L8_van_hki,L8_van_hkii,ma_hoc_sinh
0,8000180385,8.9,7.9,9.4,8.5,8.6,8.5,8.7,7.6,9.3,...,8.4,8.4,8.4,8.1,8.4,7.9,8.8,8.7,8.8,8000180385
1,8000180391,7.0,6.8,7.1,6.8,7.7,6.4,6.7,5.5,7.3,...,6.4,6.7,6.2,4.9,6.2,4.3,5.9,6.6,5.6,8000180391
2,8000180790,9.1,8.4,9.4,7.9,8.4,7.6,8.5,8.0,8.7,...,8.3,8.0,8.4,9.1,9.4,8.9,8.4,8.5,8.3,8000180790
3,8000180756,9.3,8.9,9.5,9.0,8.7,9.1,9.0,8.7,9.1,...,8.7,8.0,9.0,9.0,9.2,8.9,8.6,8.1,8.8,8000180756
4,8000180587,9.4,9.3,9.4,9.5,9.6,9.4,9.5,9.7,9.4,...,8.9,9.0,8.8,9.1,9.1,9.1,9.3,9.2,9.3,8000180587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,8000180818,8.6,7.9,8.9,8.8,8.1,9.2,7.9,7.2,8.3,...,7.4,5.9,8.1,7.0,7.3,6.8,7.8,7.6,7.9,8000180818
126,8000180545,7.4,6.9,7.6,7.6,7.0,7.9,6.7,5.3,7.4,...,7.3,6.4,7.8,5.8,4.8,6.3,6.5,6.7,6.4,8000180545
127,8000180738,8.4,8.9,8.1,7.2,7.7,6.9,7.6,7.6,7.6,...,7.2,7.5,7.1,7.2,6.5,7.5,6.0,5.7,6.2,8000180738
128,8000180550,8.8,7.8,9.3,8.8,9.1,8.6,8.4,7.9,8.6,...,7.2,6.4,7.6,6.6,6.3,6.8,7.8,7.9,7.7,8000180550



SHEET: Diem_L9
Số cột điểm được kiểm tra: 25
Tổng giá trị không hợp lệ: 130
Số học sinh có điểm không hợp lệ: 130

FAIL: Phát hiện điểm ngoài phạm vi hợp lệ.


,Số giá trị không hợp lệ,Tỷ lệ không hợp lệ (%)
ma_hoc_sinh,130,100.0


,ma_hoc_sinh,L9_cong_nghe_ca_nam,L9_cong_nghe_hki,L9_cong_nghe_hkii,L9_gdcd_ca_nam,L9_gdcd_hki,L9_gdcd_hkii,L9_khtn_ca_nam,L9_khtn_hki,L9_khtn_hkii,...,L9_tin_hoc_ca_nam,L9_tin_hoc_hki,L9_tin_hoc_hkii,L9_toan_ca_nam,L9_toan_hki,L9_toan_hkii,L9_van_ca_nam,L9_van_hki,L9_van_hkii,ma_hoc_sinh
0,8000180385,9.3,9.2,9.4,8.8,8.7,8.8,8.2,8.3,8.1,...,9.2,8.9,9.3,9.3,8.5,9.7,8.0,7.7,8.1,8000180385
1,8000180391,7.4,7.5,7.4,7.6,8.3,7.3,6.5,5.9,6.8,...,6.9,6.8,6.9,5.2,5.1,5.2,6.8,6.3,7.0,8000180391
2,8000180790,8.9,8.9,8.9,8.3,8.4,8.3,8.6,8.5,8.7,...,8.8,8.4,9.0,8.7,8.1,9.0,8.5,8.6,8.4,8000180790
3,8000180756,9.0,9.3,8.9,8.7,8.3,8.9,9.6,9.6,9.6,...,9.0,8.3,9.4,9.0,8.7,9.2,8.9,8.8,8.9,8000180756
4,8000180587,8.6,8.0,8.9,8.4,8.3,8.4,7.7,8.3,7.4,...,7.9,8.4,7.7,8.6,8.3,8.8,9.0,8.7,9.1,8000180587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,8000180818,8.1,8.0,8.2,8.0,7.8,8.1,7.2,7.1,7.2,...,7.1,6.8,7.2,6.7,5.5,7.3,6.8,7.1,6.6,8000180818
126,8000180545,7.6,8.4,7.2,7.4,7.7,7.2,6.5,6.2,6.7,...,6.8,6.6,6.9,5.9,6.2,5.8,5.4,5.1,5.6,8000180545
127,8000180738,8.4,8.7,8.3,6.8,5.9,7.2,7.0,5.7,7.7,...,7.2,7.7,7.0,6.3,5.0,6.9,5.6,5.1,5.8,8000180738
128,8000180550,9.5,8.8,9.8,9.0,9.3,8.9,8.4,8.3,8.4,...,8.2,7.3,8.7,8.8,8.5,9.0,8.0,7.6,8.2,8000180550


In [ ]:
# Kiểm tra dữ liệu không nhất quán giữa các sheet lớp 6–9

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

# 1. Kiểm tra cấu trúc cột giữa các sheet
reference_cols = set(dfs["Diem_L6"].columns)

for sheet_name, df in dfs.items():
    current_cols = set(df.columns)

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")

    print(f"Cột thiếu so với lớp 6: {reference_cols - current_cols}")
    print(f"Cột bổ sung so với lớp 6: {current_cols - reference_cols}")

# 2. Kiểm tra các giá trị lựa chọn môn học chỉ gồm 0 và 1
for sheet_name, df in dfs.items():

    invalid_choice = (
        ~df[choice_cols].isin([0, 1])
        & df[choice_cols].notna()
    )

    print(f"\n{sheet_name}:")
    print(f"Giá trị lựa chọn không thuộc {{0, 1}}: {invalid_choice.sum().sum():,}")

# 3. Kiểm tra mỗi học sinh chọn đúng 4/9 môn
for sheet_name, df in dfs.items():

    valid_binary = df[choice_cols].isin([0, 1]).all(axis=1)
    valid_four = valid_binary & df[choice_cols].sum(axis=1).eq(4)

    print(f"\n{sheet_name}:")
    print(f"Học sinh chọn hợp lệ: {valid_four.sum():,}")
    print(f"Học sinh chọn không hợp lệ: {(~valid_four).sum():,}")

# 4. Kiểm tra tính nhất quán lựa chọn môn giữa các lớp
reference = dfs["Diem_L6"].set_index(id_col)[choice_cols]

for grade in [7, 8, 9]:

    current = dfs[f"Diem_L{grade}"].set_index(id_col)[choice_cols]

    common_ids = reference.index.intersection(current.index)

    ref_values = reference.loc[common_ids]
    current_values = current.loc[common_ids]

    inconsistent = (
        ref_values.ne(current_values)
        & ~(ref_values.isna() & current_values.isna())
    ).any(axis=1)

    print(f"\nLỚP 6 SO VỚI LỚP {grade}")
    print(f"Số học sinh đối chiếu: {len(common_ids):,}")
    print(f"Số học sinh không nhất quán: {inconsistent.sum():,}")

    if inconsistent.any():
        display(
            lib.pd.DataFrame({
                id_col: common_ids[inconsistent]
            }).head(20)
        )

In [ ]:
# Kiểm tra mỗi học sinh chọn đúng 4/9 môn tự chọn

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

for sheet_name, df in dfs.items():

    # Kiểm tra giá trị chỉ thuộc {0, 1}
    valid_binary = df[choice_cols].isin([0, 1]).all(axis=1)

    # Đếm số môn được chọn của mỗi học sinh
    selected_count = df[choice_cols].sum(axis=1)

    # Điều kiện hợp lệ: đúng 4 môn và chỉ chứa 0/1
    valid_selection = valid_binary & selected_count.eq(4)

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")

    print(f"Tổng số học sinh: {len(df):,}")
    print(f"Số học sinh hợp lệ: {valid_selection.sum():,}")
    print(f"Số học sinh không hợp lệ: {(~valid_selection).sum():,}")
    print(f"Tỷ lệ hợp lệ: {valid_selection.mean() * 100:.2f}%")

    # Phân phối số môn được chọn
    print("\nPhân phối số môn được chọn:")
    display(selected_count.value_counts().sort_index().to_frame("Số học sinh"))

    # Hiển thị học sinh vi phạm
    if (~valid_selection).any():
        print("\nDanh sách học sinh không hợp lệ:")
        invalid_df = df.loc[
            ~valid_selection,
            [id_col] + choice_cols
        ].copy()

        invalid_df["so_mon_da_chon"] = selected_count.loc[
            ~valid_selection
        ]

        display(invalid_df)
    else:
        print("\nPASS: Tất cả học sinh đều chọn đúng 4/9 môn.")

In [ ]:
# Phát hiện giá trị ngoại lệ (Outliers) bằng phương pháp IQR
# Chỉ kiểm tra các cột điểm, không tự động loại bỏ dữ liệu

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

outlier_reports = {}
outlier_details = {}

for sheet_name, df in dfs.items():

    # Chỉ lấy các cột điểm số
    score_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in choice_cols and col != id_col
    ]

    # Tính Q1, Q3 và IQR
    Q1 = df[score_cols].quantile(0.25)
    Q3 = df[score_cols].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Xác định Outliers
    outlier_mask = (
        df[score_cols].lt(lower_bound)
        | df[score_cols].gt(upper_bound)
    )

    # Báo cáo theo từng cột
    report = lib.pd.DataFrame({
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Ngưỡng dưới": lower_bound,
        "Ngưỡng trên": upper_bound,
        "Số Outliers": outlier_mask.sum(),
        "Tỷ lệ Outliers (%)": (
            outlier_mask.sum() / len(df) * 100
        ).round(2)
    })

    report = report.sort_values(
        "Số Outliers", ascending=False
    )

    outlier_reports[sheet_name] = report

    # Lưu các học sinh có ít nhất một điểm ngoại lệ
    outlier_details[sheet_name] = df.loc[
        outlier_mask.any(axis=1),
        [id_col] + score_cols
    ].copy()

    print(f"\n{'=' * 60}")
    print(f"SHEET: {sheet_name}")
    print(f"{'=' * 60}")

    print(f"Số cột điểm kiểm tra: {len(score_cols)}")
    print(f"Tổng số giá trị ngoại lệ: {outlier_mask.sum().sum():,}")
    print(f"Số học sinh có ít nhất một ngoại lệ: {outlier_mask.any(axis=1).sum():,}")
    print(
        f"Tỷ lệ học sinh có ngoại lệ: "
        f"{outlier_mask.any(axis=1).mean() * 100:.2f}%"
    )

    display(report)

    if outlier_mask.any().any():
        print("\n20 học sinh đầu tiên có giá trị ngoại lệ:")
        display(outlier_details[sheet_name].head(20))
    else:
        print("PASS: Không phát hiện giá trị ngoại lệ theo IQR.")

In [ ]:
from pathlib import Path

# Thư mục lưu báo cáo
report_dir = Path("reports")
report_dir.mkdir(parents=True, exist_ok=True)

# 1. Tổng hợp báo cáo ngoại lệ
summary = lib.pd.concat(
    outlier_reports,
    names=["Sheet", "Column"]
).reset_index()

summary.to_csv(
    report_dir / "outlier_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Lưu danh sách học sinh có ngoại lệ
details = lib.pd.concat(
    outlier_details,
    names=["Sheet", "Row"]
).reset_index()

details.to_csv(
    report_dir / "outlier_details.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu báo cáo Outliers.")

In [ ]:
# ==========================================
# TRỰC QUAN HÓA OUTLIERS - IQR
# ==========================================

# 1. Thiết lập thư mục lưu biểu đồ
fig_dir = lib.Path("figures/outliers")
fig_dir.mkdir(parents=True, exist_ok=True)

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

outlier_masks = {}
outlier_rates = []
outlier_counts = {}

for sheet_name, df in dfs.items():

    score_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in choice_cols and col != "ma_hoc_sinh"
    ]

    Q1 = df[score_cols].quantile(0.25)
    Q3 = df[score_cols].quantile(0.75)
    IQR = Q3 - Q1

    mask = (
        df[score_cols].lt(Q1 - 1.5 * IQR)
        | df[score_cols].gt(Q3 + 1.5 * IQR)
    )

    outlier_masks[sheet_name] = mask
    outlier_counts[sheet_name] = mask.sum()

    outlier_rates.append({
        "Lớp": sheet_name.replace("Diem_L", "Lớp "),
        "Tỷ lệ (%)": mask.any(axis=1).mean() * 100
    })


# ==========================================
# 2. BOXPLOT - PHÂN PHỐI ĐIỂM THEO LỚP
# ==========================================

fig, axes = lib.plt.subplots(2, 2, figsize=(20, 12))

for ax, (sheet_name, df) in zip(axes.flat, dfs.items()):

    score_cols = outlier_masks[sheet_name].columns.tolist()

    lib.sns.boxplot(
        data=df[score_cols],
        ax=ax,
        color="skyblue",
        fliersize=2
    )

    ax.set_title(
        f"Phân phối điểm - {sheet_name}",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Môn học / Học kỳ")
    ax.set_ylabel("Điểm")
    ax.set_ylim(-0.5, 10.5)
    ax.tick_params(axis="x", rotation=90)

lib.plt.tight_layout()

lib.plt.savefig(
    fig_dir / "boxplot_scores.png",
    dpi=300,
    bbox_inches="tight"
)

lib.plt.show()


# ==========================================
# 3. BAR CHART - TỶ LỆ HỌC SINH CÓ OUTLIERS
# ==========================================

rate_df = lib.pd.DataFrame(outlier_rates)

fig, ax = lib.plt.subplots(figsize=(9, 5))

lib.sns.barplot(
    data=rate_df,
    x="Lớp",
    y="Tỷ lệ (%)",
    color="steelblue",
    ax=ax
)

for i, row in rate_df.iterrows():
    ax.text(
        i,
        row["Tỷ lệ (%)"] + 0.05,
        f'{row["Tỷ lệ (%)"]:.2f}%',
        ha="center",
        fontsize=11
    )

ax.set_title(
    "Tỷ lệ học sinh có ít nhất một giá trị ngoại lệ",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel("Tỷ lệ học sinh (%)")
ax.set_xlabel("Khối lớp")
ax.set_ylim(0, rate_df["Tỷ lệ (%)"].max() * 1.2)

lib.plt.tight_layout()

lib.plt.savefig(
    fig_dir / "outlier_rate_by_grade.png",
    dpi=300,
    bbox_inches="tight"
)

lib.plt.show()


# ==========================================
# 4. HEATMAP - SỐ LƯỢNG OUTLIERS THEO CỘT
# ==========================================

heatmap_df = lib.pd.DataFrame(outlier_counts).fillna(0).astype(int)

heatmap_df.columns = [
    col.replace("Diem_L", "Lớp ")
    for col in heatmap_df.columns
]

fig, ax = lib.plt.subplots(figsize=(10, 12))

lib.sns.heatmap(
    heatmap_df,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Số giá trị ngoại lệ"},
    ax=ax
)

ax.set_title(
    "Số lượng giá trị ngoại lệ theo môn học và khối lớp",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel("Khối lớp")
ax.set_ylabel("Cột điểm")

lib.plt.tight_layout()

lib.plt.savefig(
    fig_dir / "outlier_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

lib.plt.show()

print(f"Đã lưu 3 biểu đồ tại: {fig_dir.resolve()}")